# Caracterización Interactiva y Extracción .OP de Transistores Sky130 (Std y LVT)
---
Este cuaderno realiza la caracterización analógica completa y extracción del punto de operación (.OP) para los 4 transistores clave del proceso Sky130 (`Mplvt`, `Mp`, `Mnlvt`, `Mn`), utilizando el lector nativo en Python estándar de archivos RAW ASCII de NGspice:

1. **Simulación Dinámica en NGspice y Lector Nativo en Python Estándar (`read_ngspice_raw_ascii`):**
   - Actualización de parámetros geométricos ($W, L, \text{nf}, \text{mult}$) y tensiones de polarización ($V_{SG}, V_{GS}$) directamente desde Python.
   - Ajuste de $V_{SG} = 1.20\text{ V}$ para la conducción nominal objetivo de $I_d \approx 10\mu\text{A}$ en el transistor de referencia $M_p$ ($W=4\mu\text{m}, L=0.5\mu\text{m}$).
   - Ejecución en batch de NGspice y carga directa sin librerías externas mediante `read_ngspice_raw_ascii()`.

2. **Extracción Completa de la Tabla de Punto de Operación (.OP):**
   - Cálculo de corriente total ($I_{D,\text{total}}$), capacitancia total ($C_{gg,\text{total}}$), transconductancia total ($g_{m,\text{total}}$), resistencia de salida ($r_o$) y frecuencia de transición ($f_T$).
   - Verificación estricta de las regiones de **Inversión** (Débil, Moderada, Fuerte según $\Delta_{ov} = V_{gs} - V_{th}$) y **Operación** (Saturación o Triodo según $\Delta_{vd} = V_{ds} - V_{dsat}$).

3. **Curvas de Caracterización Analógica Interactivas en Plotly:**
   - Subgráficos de $I_{D,\text{total}}$, $g_m/I_D$, $f_T$, $r_o$ y $V_{dsat}$ vs $V_{SG}$ (PMOS) y $V_{GS}$ (NMOS) con escala logarítmica explícita para resistencia de salida.


In [1]:
# ==============================================================================
# Encabezado: Simulación en NGspice y Lector Nativo RAW ASCII en Python Estándar
# Descripción: Modifica dinámicamente las dimensiones en el netlist SPICE, ejecuta
#              la simulación en NGspice y lee el archivo RAW usando Python nativo.
# ==============================================================================

import os
import re
import subprocess
import numpy as np

# ── 1. Parámetros Configurables de Polarización y Geometría ───────────────────
VSG_TARGET = 1.20  # Tensión VSG objetivo para evaluar PMOS (conducción de 10 uA en Mp) [V]
VGS_TARGET = 1.00  # Tensión VGS objetivo para evaluar NMOS [V]

# Dimensiones geométricas, dedos (nf) y multiplicidad (mult)
W_MPLVT = 1000; L_MPLVT = 0.35; NF_MPLVT = 50; MULT_MPLVT = 5
W_MP    = 4.0;  L_MP    = 0.50; NF_MP    = 1;  MULT_MP    = 1
W_MNLVT = 1.0;  L_MNLVT = 0.35; NF_MNLVT = 1;  MULT_MNLVT = 1
W_MN    = 10.0; L_MN    = 0.50; NF_MN    = 1;  MULT_MN    = 1

# Rangos de barrido DC para Vgp (PMOS) y Vgn (NMOS)
VGP_START, VGP_STOP, VGP_STEP = 0.0, 1.8, 0.01
VGN_START, VGN_STOP, VGN_STEP = 0.0, 1.8, 0.01

# ── 2. Lectura y Modificación Dinámica del Netlist SPICE ──────────────────────
BASE_NETLIST = '/headless/.xschem/simulations/caracterizador.spice'
SIM_NETLIST  = '/headless/.xschem/simulations/caracterizador_sim.spice'
RAW_FILE     = '/headless/.xschem/simulations/caracterizador.raw'

if not os.path.exists(BASE_NETLIST):
    BASE_NETLIST = '../xschem/caracterizador.spice'
    SIM_NETLIST  = '../xschem/caracterizador_sim.spice'
    RAW_FILE     = '../xschem/caracterizador.raw'

with open(BASE_NETLIST, 'r') as f:
    spice_content = f.read()

def update_instance_full(text, inst_name, w_val, l_val, nf_val, mult_val):
    pattern_w_l = r'(' + inst_name + r'\s+.*?L=)[0-9.eE+-]+(\s+W=)[0-9.eE+-]+'
    text = re.sub(pattern_w_l, r'\g<1>' + str(l_val) + r'\g<2>' + str(w_val), text, flags=re.IGNORECASE)
    
    pattern_nf = r'(' + inst_name + r'\s+.*?nf=)[0-9.eE+-]+'
    text = re.sub(pattern_nf, r'\g<1>' + str(nf_val), text, flags=re.IGNORECASE)
    
    pattern_mult = r'(' + inst_name + r'\s+.*?mult=)[0-9.eE+-]+'
    text = re.sub(pattern_mult, r'\g<1>' + str(mult_val), text, flags=re.IGNORECASE)
    return text

spice_content = update_instance_full(spice_content, 'XMplvt', W_MPLVT, L_MPLVT, NF_MPLVT, MULT_MPLVT)
spice_content = update_instance_full(spice_content, 'XMp', W_MP, L_MP, NF_MP, MULT_MP)
spice_content = update_instance_full(spice_content, 'XMnlvt', W_MNLVT, L_MNLVT, NF_MNLVT, MULT_MNLVT)
spice_content = update_instance_full(spice_content, 'XMn', W_MN, L_MN, NF_MN, MULT_MN)

spice_content = re.sub(r'\.param vsg\s*=\s*[0-9.eE+-]+', f'.param vsg = {VSG_TARGET}', spice_content)
spice_content = re.sub(r'\.param vgs\s*=\s*[0-9.eE+-]+', f'.param vgs = {VGS_TARGET}', spice_content)

spice_content = re.sub(
    r'dc Vgp\s+[0-9.eE+-]+\s+[0-9.eE+-]+\s+[0-9.eE+-]+\s+Vgn\s+[0-9.eE+-]+\s+[0-9.eE+-]+\s+[0-9.eE+-]+',
    f'dc Vgp {VGP_START} {VGP_STOP} {VGP_STEP} Vgn {VGN_START} {VGN_STOP} {VGN_STEP}',
    spice_content
)

with open(SIM_NETLIST, 'w') as f:
    f.write(spice_content)

print(f"Netlist actualizado en: {SIM_NETLIST}")

# ── 3. Ejecución de Simulación en NGspice ─────────────────────────────────────
CONTAINER = "iic-osic-tools_chipathon_xvnc_uid_1000"
sim_dir = os.path.dirname(SIM_NETLIST)
sim_file = os.path.basename(SIM_NETLIST)
cmd_sim = f'export PATH=/foss/tools/xschem/bin:/foss/tools/ngspice/bin:$PATH; cd {sim_dir} && ngspice -b {sim_file}'
result = subprocess.run(f'docker exec {CONTAINER} bash -c "{cmd_sim}"', shell=True, capture_output=True, text=True)

if result.returncode == 0:
    print("¡Simulación en NGspice completada con éxito!")
else:
    print("Salida de NGspice:", result.stdout)

# ── 4. Lector Nativo RAW ASCII de NGspice en Python Estándar ───────────────────
def read_ngspice_raw_ascii(filepath):
    with open(filepath, 'r') as f:
        lines = f.readlines()
        
    num_vars = 0
    is_complex = False
    var_names = []
    idx = 0
    
    while idx < len(lines):
        line = lines[idx].strip()
        if line.startswith('Flags:'):
            if 'complex' in line.lower():
                is_complex = True
        elif line.startswith('No. Variables:'):
            num_vars = int(line.split(':')[1].strip())
        elif line.startswith('Variables:'):
            idx += 1
            for _ in range(num_vars):
                parts = lines[idx].strip().split()
                var_names.append(parts[1])
                idx += 1
            continue
        elif line.startswith('Values:'):
            idx += 1
            break
        idx += 1
        
    data = {v: [] for v in var_names}
    current_pt = 0
    
    while idx < len(lines):
        line = lines[idx].strip()
        if not line:
            idx += 1
            continue
        parts = line.split('\t')
        val_str = parts[1].strip() if len(parts) >= 2 else line
        
        if is_complex:
            if ',' in val_str:
                r, i = map(float, val_str.split(','))
                val = complex(r, i)
            else:
                val = complex(float(val_str), 0.0)
        else:
            val = float(val_str)
            
        var_name = var_names[current_pt % num_vars]
        data[var_name].append(val)
        current_pt += 1
        idx += 1
        
    for k in data:
        data[k] = np.array(data[k])
    return data

raw_file_path = RAW_FILE if os.path.exists(RAW_FILE) else "/foss/designs/LDO-sky130/xschem/caracterizador.raw"
raw_data = read_ngspice_raw_ascii(raw_file_path)
print(f"¡Carga exitosa con lector nativo! Total de variables cargadas: {len(raw_data)}")

Netlist actualizado en: /headless/.xschem/simulations/caracterizador_sim.spice
Salida de NGspice: 


¡Carga exitosa con lector nativo! Total de variables cargadas: 63


In [2]:
# ==============================================================================
# Encabezado: Extracción de Tabla .OP con Geometría, Criterios de Inversión y Saturación
# Descripción: Extrae parámetros del punto de operación, calcula corriente y capacitancia
#              totales, y evalúa los criterios de Inversión (Δov = Vgs - Vth) y 
#              Saturación (Δvd = Vds - Vdsat) para cada transistor.
# ==============================================================================

import pandas as pd

# Slices para separar los dos barridos DC (181 puntos cada uno)
p_slice = slice(0, 181)
n_slice = slice(0, None, 181)

vgs_p_sweep = np.real(np.abs(raw_data['v(@m.xmp.msky130_fd_pr__pfet_01v8[vgs])'][p_slice]))
vgs_n_sweep = np.real(np.abs(raw_data['v(@m.xmn.msky130_fd_pr__nfet_01v8[vgs])'][n_slice]))

idx_p = int(np.argmin(np.abs(vgs_p_sweep - VSG_TARGET)))
idx_n = int(np.argmin(np.abs(vgs_n_sweep - VGS_TARGET)))

transistores = {
    'Mplvt': {'tipo': 'PMOS', 'slice': p_slice, 'idx': idx_p, 'w': W_MPLVT, 'l': L_MPLVT, 'nf': NF_MPLVT, 'mult': MULT_MPLVT, 'gm': '@m.xmplvt.msky130_fd_pr__pfet_01v8_lvt[gm]', 'gds': '@m.xmplvt.msky130_fd_pr__pfet_01v8_lvt[gds]', 'id': 'i(@m.xmplvt.msky130_fd_pr__pfet_01v8_lvt[id])', 'vth': 'v(@m.xmplvt.msky130_fd_pr__pfet_01v8_lvt[vth])', 'vdsat': 'v(@m.xmplvt.msky130_fd_pr__pfet_01v8_lvt[vdsat])', 'cgg': '@m.xmplvt.msky130_fd_pr__pfet_01v8_lvt[cgg]', 'vgs': 'v(@m.xmplvt.msky130_fd_pr__pfet_01v8_lvt[vgs])'},
    'Mp':    {'tipo': 'PMOS', 'slice': p_slice, 'idx': idx_p, 'w': W_MP,    'l': L_MP,    'nf': NF_MP,    'mult': MULT_MP,    'gm': '@m.xmp.msky130_fd_pr__pfet_01v8[gm]',        'gds': '@m.xmp.msky130_fd_pr__pfet_01v8[gds]',        'id': 'i(@m.xmp.msky130_fd_pr__pfet_01v8[id])',        'vth': 'v(@m.xmp.msky130_fd_pr__pfet_01v8[vth])',        'vdsat': 'v(@m.xmp.msky130_fd_pr__pfet_01v8[vdsat])',        'cgg': '@m.xmp.msky130_fd_pr__pfet_01v8[cgg]',        'vgs': 'v(@m.xmp.msky130_fd_pr__pfet_01v8[vgs])'},
    'Mnlvt': {'tipo': 'NMOS', 'slice': n_slice, 'idx': idx_n, 'w': W_MNLVT, 'l': L_MNLVT, 'nf': NF_MNLVT, 'mult': MULT_MNLVT, 'gm': '@m.xmnlvt.msky130_fd_pr__nfet_01v8_lvt[gm]', 'gds': '@m.xmnlvt.msky130_fd_pr__nfet_01v8_lvt[gds]', 'id': 'i(@m.xmnlvt.msky130_fd_pr__nfet_01v8_lvt[id])', 'vth': 'v(@m.xmnlvt.msky130_fd_pr__nfet_01v8_lvt[vth])', 'vdsat': 'v(@m.xmnlvt.msky130_fd_pr__nfet_01v8_lvt[vdsat])', 'cgg': '@m.xmnlvt.msky130_fd_pr__nfet_01v8_lvt[cgg]', 'vgs': 'v(@m.xmnlvt.msky130_fd_pr__nfet_01v8_lvt[vgs])'},
    'Mn':    {'tipo': 'NMOS', 'slice': n_slice, 'idx': idx_n, 'w': W_MN,    'l': L_MN,    'nf': NF_MN,    'mult': MULT_MN,    'gm': '@m.xmn.msky130_fd_pr__nfet_01v8[gm]',        'gds': '@m.xmn.msky130_fd_pr__nfet_01v8[gds]',        'id': 'i(@m.xmn.msky130_fd_pr__nfet_01v8[id])',        'vth': 'v(@m.xmn.msky130_fd_pr__nfet_01v8[vth])',        'vdsat': 'v(@m.xmn.msky130_fd_pr__nfet_01v8[vdsat])',        'cgg': '@m.xmn.msky130_fd_pr__nfet_01v8[cgg]',        'vgs': 'v(@m.xmn.msky130_fd_pr__nfet_01v8[vgs])'}
}

tabla_op = []
for nombre, info in transistores.items():
    s = info['slice']
    idx = info['idx']
    m_factor = info['mult']
    
    gm_unit = float(np.real(np.abs(raw_data[info['gm']][s][idx])))
    gds_unit = float(np.real(np.abs(raw_data[info['gds']][s][idx])))
    id_unit = float(np.real(np.abs(raw_data[info['id']][s][idx])))
    vth = float(np.real(raw_data[info['vth']][s][idx]))
    vdsat = float(np.real(raw_data[info['vdsat']][s][idx]))
    cgg_unit = float(np.real(np.abs(raw_data[info['cgg']][s][idx])))
    vgs = float(np.real(np.abs(raw_data[info['vgs']][s][idx])))
    vds = 1.80  # Tensión VDS / VSD aplicada constante
    
    id_total = id_unit * m_factor
    gm_total = gm_unit * m_factor
    gds_total = gds_unit * m_factor
    cgg_total = cgg_unit * m_factor
    
    ro_total = 1.0 / gds_total if gds_total != 0 else np.nan
    gm_id = gm_unit / id_unit if id_unit != 0 else np.nan
    gm_ro = gm_total * ro_total
    ft = gm_unit / (2 * np.pi * cgg_unit) if cgg_unit != 0 else np.nan
    
    # ── Criterios de Inversión y Saturación ───────────────────────────────────
    delta_ov = vgs - abs(vth)   # Δov = Vgs - Vth
    delta_vd = vds - vdsat      # Δvd = Vds - Vdsat
    
    if delta_ov < 0:
        inv_class = "Inversión Débil (Weak)"
    elif delta_ov <= 0.10:
        inv_class = "Inversión Moderada"
    else:
        inv_class = "Inversión Fuerte (Strong)"
        
    if delta_vd > 0:
        sat_class = "Saturación"
    else:
        sat_class = "Triodo"

    tabla_op.append({
        'Transistor': nombre,
        'W (um)': info['w'],
        'L (um)': info['l'],
        'nf': info['nf'],
        'mult': info['mult'],
        'Id_total (uA)': round(id_total * 1e6, 2),
        'Cgg_total (fF)': round(cgg_total * 1e15, 2),
        'Vgs (V)': round(vgs, 2),
        'Vds (V)': round(vds, 2),
        'Vth (V)': round(vth, 3),
        'Vdsat (V)': round(vdsat, 3),
        'Δov = Vgs-Vth (V)': round(delta_ov, 3),
        'Δvd = Vds-Vdsat (V)': round(delta_vd, 3),
        'Inversión': inv_class,
        'Región': sat_class,
        'gm_total (uS)': round(gm_total * 1e6, 2),
        'ro_total (kOhm)': round(ro_total / 1e3, 2),
        'gm*ro (V/V)': round(gm_ro, 2),
        'gm/Id (1/V)': round(gm_id, 2),
        'fT (GHz)': round(ft / 1e9, 2)
    })

df_op = pd.DataFrame(tabla_op)
print(f"=== TABLA DE PUNTO DE OPERACIÓN (.OP) COMPLETA (Evaluado a VSG={vgs_p_sweep[idx_p]:.2f}V / VGS={vgs_n_sweep[idx_n]:.2f}V) ===")
display(df_op)

=== TABLA DE PUNTO DE OPERACIÓN (.OP) COMPLETA (Evaluado a VSG=1.20V / VGS=1.00V) ===


,Transistor,W (um),L (um),nf,mult,Id_total (uA),Cgg_total (fF),Vgs (V),Vds (V),Vth (V),Vdsat (V),Δov = Vgs-Vth (V),Δvd = Vds-Vdsat (V),Inversión,Región,gm_total (uS),ro_total (kOhm),gm*ro (V/V),gm/Id (1/V),fT (GHz)
0,Mplvt,1000.0,0.35,50,5,79663.68,2267.10,1.2,1.8,0.348,0.626,0.852,1.174,Inversión Fuerte (Strong),Saturación,162463.71,0.07,10.65,2.04,11.41
1,Mp,4.0,0.50,1,1,9.80,5.55,1.2,1.8,0.954,0.249,0.246,1.551,Inversión Fuerte (Strong),Saturación,69.78,1772.26,123.66,7.12,2.00
2,Mnlvt,1.0,0.35,1,1,56.00,2.65,1.0,1.8,0.531,0.363,0.469,1.437,Inversión Fuerte (Strong),Saturación,208.08,421.54,87.72,3.72,12.50
3,Mn,10.0,0.50,1,1,32.28,2.89,1.0,1.8,0.634,0.279,0.366,1.521,Inversión Fuerte (Strong),Saturación,158.51,591.24,93.72,4.91,8.73


In [3]:
# ==============================================================================
# Encabezado: Curvas de Caracterización Analógica Interactivas (Plotly)
# Descripción: Muestra gráficos interactivos de Id_total, gm/Id, fT, ro, Vdsat 
#              vs VGS (o VSG), separados claramente para PMOS y NMOS.
# ==============================================================================

import plotly.graph_objects as go
from plotly.subplots import make_subplots

fig = make_subplots(
    rows=2, cols=5,
    subplot_titles=(
        "PMOS: Id_total vs VSG", "PMOS: gm/Id vs VSG", "PMOS: fT vs VSG", "PMOS: ro vs VSG", "PMOS: Vdsat vs VSG",
        "NMOS: Id_total vs VGS", "NMOS: gm/Id vs VGS", "NMOS: fT vs VGS", "NMOS: ro vs VGS", "NMOS: Vdsat vs VGS"
    )
)

colores = {'Mplvt': '#1f77b4', 'Mp': '#ff7f0e', 'Mnlvt': '#2ca02c', 'Mn': '#d62728'}

for nombre, info in transistores.items():
    s = info['slice']
    m_factor = info['mult']
    
    vgs = np.real(np.abs(raw_data[info['vgs']][s]))
    id_tot = np.real(np.abs(raw_data[info['id']][s])) * m_factor * 1e6  # uA
    gm = np.real(np.abs(raw_data[info['gm']][s]))
    gds = np.real(np.abs(raw_data[info['gds']][s]))
    id_u = np.real(np.abs(raw_data[info['id']][s]))
    vdsat = np.real(raw_data[info['vdsat']][s])
    
    gm_id = np.where(id_u > 0, gm / id_u, np.nan)
    cgg_u = np.real(np.abs(raw_data[info['cgg']][s]))
    ft = np.where(cgg_u > 0, gm / (2 * np.pi * cgg_u) / 1e9, np.nan) # GHz
    ro = np.where(gds > 0, (1.0 / gds) / 1e3, np.nan)               # kOhm
    
    row = 1 if info['tipo'] == 'PMOS' else 2
    c = colores[nombre]
    
    # 1. Id_total vs VGS/VSG
    fig.add_trace(go.Scatter(x=vgs, y=id_tot, mode='lines', name=nombre, line=dict(color=c, width=2), legendgroup=nombre), row=row, col=1)
    # 2. gm/Id vs VGS/VSG
    fig.add_trace(go.Scatter(x=vgs, y=gm_id, mode='lines', name=nombre, line=dict(color=c, width=2), legendgroup=nombre, showlegend=False), row=row, col=2)
    # 3. fT vs VGS/VSG
    fig.add_trace(go.Scatter(x=vgs, y=ft, mode='lines', name=nombre, line=dict(color=c, width=2), legendgroup=nombre, showlegend=False), row=row, col=3)
    # 4. ro vs VGS/VSG
    fig.add_trace(go.Scatter(x=vgs, y=ro, mode='lines', name=nombre, line=dict(color=c, width=2), legendgroup=nombre, showlegend=False), row=row, col=4)
    # 5. Vdsat vs VGS/VSG
    fig.add_trace(go.Scatter(x=vgs, y=vdsat, mode='lines', name=nombre, line=dict(color=c, width=2), legendgroup=nombre, showlegend=False), row=row, col=5)

# Actualizar ejes
for col in range(1, 6):
    fig.update_xaxes(title_text="VSG [V]", row=1, col=col)
    fig.update_xaxes(title_text="VGS [V]", row=2, col=col)

fig.update_yaxes(title_text="Id_total [uA]", row=1, col=1)
fig.update_yaxes(title_text="gm/Id [1/V]", row=1, col=2)
fig.update_yaxes(title_text="fT [GHz]", row=1, col=3)
fig.update_yaxes(title_text="ro [kOhm]", type="log", row=1, col=4)
fig.update_yaxes(title_text="Vdsat [V]", row=1, col=5)

fig.update_yaxes(title_text="Id_total [uA]", row=2, col=1)
fig.update_yaxes(title_text="gm/Id [1/V]", row=2, col=2)
fig.update_yaxes(title_text="fT [GHz]", row=2, col=3)
fig.update_yaxes(title_text="ro [kOhm]", type="log", row=2, col=4)
fig.update_yaxes(title_text="Vdsat [V]", row=2, col=5)

fig.update_layout(height=750, title_text="<b>Caracterización Analógica Completa (Id_total, gm/Id, fT, ro, Vdsat vs VGS/VSG)</b>", template="plotly_white")
fig.show()